# Resolve Wikipedia references for explorer antibiotics

Look up every antibiotic name present in `sites/amr-si-explorer/dist/pairs.json` and save an explicit association table to `sites/amr-si-explorer/data/antibiotic_wikipedia_links.csv`. Exact Wikipedia titles and redirects are preferred. When those do not resolve, the table records a Wikipedia title-search candidate for review. The lookup is resumable; set `FORCE_REFRESH = True` to redo completed rows.

The builder also writes `sites/amr-si-explorer/dist/antibiotic_wikipedia_links.json`, which the static explorer uses for the selected antibiotic’s Wikipedia link.

In [1]:
import json
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sites/amr-si-explorer/dist/pairs.json").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Antimicrobial-Resistance repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent

PAIR_DATA_PATH = PROJECT_ROOT / "sites/amr-si-explorer/dist/pairs.json"
LOOKUP_TABLE_PATH = PROJECT_ROOT / "sites/amr-si-explorer/data/antibiotic_wikipedia_links.csv"
SITE_DATA_PATH = PROJECT_ROOT / "sites/amr-si-explorer/dist/antibiotic_wikipedia_links.json"
BUILDER_SCRIPT = PROJECT_ROOT / "sites/amr-si-explorer/scripts/build_antibiotic_wikipedia_links.py"
FORCE_REFRESH = False

pair_records = json.loads(PAIR_DATA_PATH.read_text(encoding="utf-8"))["records"]
antibiotic_count = len({record["antibiotic"] for record in pair_records})
existing_count = len(pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False)) if LOOKUP_TABLE_PATH.exists() else 0
display(Markdown(
    f"**Lookup scope:** {antibiotic_count:,} explorer antibiotics · "
    f"**Existing associations:** {existing_count:,} rows · **Force refresh:** {FORCE_REFRESH}"
))

**Lookup scope:** 68 explorer antibiotics · **Existing associations:** 68 rows · **Force refresh:** False

## Scan Wikipedia and save the association table

The builder queries the English Wikipedia API, follows title normalization and redirects, and saves progress to CSV so completed mappings can be reused if the lookup is interrupted.

In [2]:
command = [sys.executable, str(BUILDER_SCRIPT)]
if FORCE_REFRESH:
    command.append("--force-refresh")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

Resolving 0 antibiotics; reusing 68 existing rows.
Wrote sites/amr-si-explorer/data/antibiotic_wikipedia_links.csv: 68 antibiotics, 68 exact/normalized links, 0 search candidates.


CompletedProcess(args=['/Users/f.costa/.venvs/py312/bin/python', '/Users/f.costa/Code/Antimicrobial-Resistance/sites/amr-si-explorer/scripts/build_antibiotic_wikipedia_links.py'], returncode=0)

In [3]:
antibiotic_links = pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False)
found = antibiotic_links["match_status"].eq("found").sum()
review = antibiotic_links["match_status"].eq("review_search_result").sum()
display(Markdown(
    f"**Saved:** `{LOOKUP_TABLE_PATH.relative_to(PROJECT_ROOT)}` · "
    f"**Antibiotics:** {len(antibiotic_links):,} · **Exact or normalized pages:** {found:,} · "
    f"**Search candidates to review:** {review:,}"
))
display(antibiotic_links)

**Saved:** `sites/amr-si-explorer/data/antibiotic_wikipedia_links.csv` · **Antibiotics:** 68 · **Exact or normalized pages:** 68 · **Search candidates to review:** 0

,antibiotic,wikipedia_title,wikipedia_url,match_status,match_type
0,Amikacin,Amikacin,https://en.wikipedia.org/wiki/Amikacin,found,exact_title_or_redirect
1,Amoxicillin,Amoxicillin,https://en.wikipedia.org/wiki/Amoxicillin,found,exact_title_or_redirect
2,Amoxicillin/clavulanic-acid,Amoxicillin/clavulanic acid,https://en.wikipedia.org/wiki/Amoxicillin%2Fcl...,found,normalized_title_or_redirect
3,Ampicillin,Ampicillin,https://en.wikipedia.org/wiki/Ampicillin,found,exact_title_or_redirect
4,Ampicillin/sulbactam,Ampicillin/sulbactam,https://en.wikipedia.org/wiki/Ampicillin%2Fsul...,found,exact_title_or_redirect
...,...,...,...,...,...
63,Ticarcillin,Ticarcillin,https://en.wikipedia.org/wiki/Ticarcillin,found,exact_title_or_redirect
64,Tigecycline,Tigecycline,https://en.wikipedia.org/wiki/Tigecycline,found,exact_title_or_redirect
65,Tobramycin,Tobramycin,https://en.wikipedia.org/wiki/Tobramycin,found,exact_title_or_redirect
66,Trimethoprim,Trimethoprim,https://en.wikipedia.org/wiki/Trimethoprim,found,exact_title_or_redirect
